# PyTorch 猫狗分类实验

这个 Notebook 使用项目中的 PyTorch 代码，按顺序展示数据、模型、训练记录、混淆矩阵、误判样本和卷积核。

模型结构：`Conv → ReLU → Pool → Conv → ReLU → Pool → Linear → 猫/狗`。

## 0. 使用说明

1. 在项目根目录启动：`jupyter notebook notebooks/cat_dog_classifier_workflow.ipynb`。
2. 从上到下运行即可，默认读取已经训练好的模型，不会重新训练或覆盖模型。
3. 想在 Notebook 中训练时，将 `RUN_TRAINING` 改为 `True`。
4. Mac 优先使用 MPS，没有可用 GPU 时自动使用 CPU。

In [ ]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch import nn


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'implementations' / 'pytorch').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录，请从猫狗分类器目录启动 Notebook')


PROJECT_ROOT = find_project_root()
PYTORCH_DIR = PROJECT_ROOT / 'implementations' / 'pytorch'
if str(PYTORCH_DIR) not in sys.path:
    sys.path.insert(0, str(PYTORCH_DIR))

from data import build_test_loader, build_train_validation_loaders
from engine import evaluate, train_one_epoch
from model import CatDogCNN
from utils import load_checkpoint, select_device, set_seed

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = [
    'PingFang SC', 'Heiti SC', 'Arial Unicode MS', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

DEVICE = select_device('auto')
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'pytorch'
MODEL_PATH = OUTPUT_DIR / 'best_model_pytorch.pt'
print(f'项目目录：{PROJECT_ROOT}')
print(f'PyTorch：{torch.__version__}')
print(f'运行设备：{DEVICE}')

## 1. 查看数据

先随机查看训练图片，再统计训练集和测试集的猫狗数量。

In [ ]:
train_root = PROJECT_ROOT / 'data' / 'processed' / 'train'
rng = np.random.default_rng(23)
sample_paths = []
for class_name in ('cats', 'dogs'):
    paths = sorted((train_root / class_name).glob('*.jpg'))
    indices = rng.choice(len(paths), size=4, replace=False)
    sample_paths.extend(paths[index] for index in indices)

figure, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for axis, image_path in zip(axes.flat, sample_paths):
    with Image.open(image_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(f'{image_path.parent.name[:-1]}：{image_path.name}', fontsize=9)
    axis.axis('off')
figure.suptitle('训练图片示例')
figure.tight_layout()
plt.show()

In [ ]:
def count_images(split):
    root = PROJECT_ROOT / 'data' / 'processed' / split
    return [len(list((root / name).glob('*.jpg'))) for name in ('cats', 'dogs')]

train_counts = count_images('train')
test_counts = count_images('test')
x = np.arange(2)
width = 0.34

figure, axis = plt.subplots(figsize=(7, 4))
axis.bar(x - width / 2, train_counts, width, label='训练目录')
axis.bar(x + width / 2, test_counts, width, label='测试目录')
axis.set_xticks(x, ('猫', '狗'))
axis.set_ylabel('图片数量')
axis.set_title('数据集类别分布')
axis.legend()
axis.bar_label(axis.containers[0])
axis.bar_label(axis.containers[1])
figure.tight_layout()
plt.show()

## 2. 加载训练好的模型

模型有两层卷积。这里读取验证准确率最高的参数，并统计参数量。

In [ ]:
checkpoint = load_checkpoint(MODEL_PATH)
model = CatDogCNN().to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(model)
print(f'最佳模型轮次：{checkpoint["epoch"]}')
print(f'最佳验证准确率：{checkpoint["validation_accuracy"] * 100:.2f}%')
print(f'总参数量：{total_parameters:,}')
print(f'可训练参数量：{trainable_parameters:,}')

## 3. 查看第一层特征图

第一层卷积会把一张 RGB 图片转换成 8 张特征图。亮的位置表示该卷积核在这里产生了较强响应。

In [ ]:
train_loader, validation_loader, class_to_idx = build_train_validation_loaders(
    data_dir=train_root,
    batch_size=64,
    validation_ratio=0.2,
    seed=23,
    num_workers=0,
    pin_memory=DEVICE.type == 'cuda',
)
images, labels = next(iter(validation_loader))

with torch.inference_mode():
    feature_maps = model.features[:2](images[:1].to(DEVICE)).cpu()[0]

figure = plt.figure(figsize=(11, 6))
grid = figure.add_gridspec(2, 5)
image_axis = figure.add_subplot(grid[:, 0])
display_image = images[0].permute(1, 2, 0).numpy() * 0.5 + 0.5
image_axis.imshow(np.clip(display_image, 0, 1))
image_axis.set_title(f'输入：{"猫" if labels[0].item() == 0 else "狗"}')
image_axis.axis('off')

for index in range(8):
    row, column = divmod(index, 4)
    axis = figure.add_subplot(grid[row, column + 1])
    axis.imshow(feature_maps[index], cmap='viridis')
    axis.set_title(f'Feature {index + 1}', fontsize=9)
    axis.axis('off')
figure.suptitle('第一层卷积后的特征图')
figure.tight_layout()
plt.show()

## 4. 训练

默认读取已有训练记录。把 `RUN_TRAINING` 改为 `True` 后会在内存中重新训练，不会覆盖原模型文件。先把 `EPOCHS` 设小一点确认流程正常。

In [ ]:
RUN_TRAINING = False
EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

if RUN_TRAINING:
    set_seed(23)
    model = CatDogCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    history = []

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_accuracy = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE
        )
        validation_loss, validation_accuracy, _ = evaluate(
            model, validation_loader, criterion, DEVICE
        )
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'validation_loss': validation_loss,
            'train_accuracy': train_accuracy,
            'validation_accuracy': validation_accuracy,
        })
        print(
            f'Epoch {epoch}: train loss={train_loss:.4f}, '
            f'val loss={validation_loss:.4f}, '
            f'val acc={validation_accuracy * 100:.2f}%'
        )
else:
    history_path = OUTPUT_DIR / 'training_history.json'
    history = json.loads(history_path.read_text(encoding='utf-8'))
    print(f'读取已有训练记录：{history_path}')

In [ ]:
epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
validation_loss = [row['validation_loss'] for row in history]
train_accuracy = [row['train_accuracy'] * 100 for row in history]
validation_accuracy = [row['validation_accuracy'] * 100 for row in history]

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, train_loss, marker='o', label='训练')
axes[0].plot(epochs, validation_loss, marker='o', label='验证')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='训练与验证损失')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(epochs, train_accuracy, marker='o', label='训练')
axes[1].plot(epochs, validation_accuracy, marker='o', label='验证')
axes[1].set(xlabel='Epoch', ylabel='准确率（%）', title='训练与验证准确率')
axes[1].grid(alpha=0.25)
axes[1].legend()
figure.tight_layout()
plt.show()

## 5. 测试结果与混淆矩阵

默认读取现有测试结果。需要重新测试当前内存中的模型时，把 `RUN_TEST_EVALUATION` 改为 `True`。

In [ ]:
RUN_TEST_EVALUATION = False

if RUN_TEST_EVALUATION:
    test_loader, _ = build_test_loader(
        PROJECT_ROOT / 'data' / 'processed' / 'test',
        batch_size=64,
        num_workers=0,
        pin_memory=DEVICE.type == 'cuda',
    )
    test_loss, test_accuracy, matrix_tensor = evaluate(
        model, test_loader, nn.CrossEntropyLoss(), DEVICE,
        with_confusion_matrix=True,
    )
    matrix = matrix_tensor.numpy()
    print(f'测试 loss：{test_loss:.4f}，准确率：{test_accuracy * 100:.2f}%')
else:
    summary = json.loads(
        (OUTPUT_DIR / 'evaluation_summary.json').read_text(encoding='utf-8')
    )
    matrix = np.array([
        [2500 - summary['cat_as_dog'], summary['cat_as_dog']],
        [summary['dog_as_cat'], 2500 - summary['dog_as_cat']],
    ])
    test_accuracy = summary['accuracy']
    print(f'读取已有测试结果，准确率：{test_accuracy * 100:.2f}%')

percentages = matrix / matrix.sum(axis=1, keepdims=True) * 100
figure, axis = plt.subplots(figsize=(5.5, 4.8))
image = axis.imshow(percentages, cmap='Blues', vmin=0, vmax=100)
axis.set_xticks((0, 1), ('猫', '狗'))
axis.set_yticks((0, 1), ('猫', '狗'))
axis.set_xlabel('预测类别')
axis.set_ylabel('真实类别')
axis.set_title(f'混淆矩阵（准确率 {test_accuracy * 100:.2f}%）')
for row in range(2):
    for column in range(2):
        axis.text(
            column, row, f'{matrix[row, column]}\n{percentages[row, column]:.1f}%',
            ha='center', va='center',
            color='white' if percentages[row, column] > 50 else 'black',
        )
figure.colorbar(image, ax=axis, label='真实类别内比例（%）')
figure.tight_layout()
plt.show()

## 6. 典型误判样本

下面选取异常图片、低光、局部特写、遮挡和外观相似等不同情况，而不是只展示置信度最高的图片。

In [ ]:
selected_paths = (
    'data/processed/test/cats/cat.4688.jpg',
    'data/processed/test/dogs/dog.8898.jpg',
    'data/processed/test/dogs/dog.7602.jpg',
    'data/processed/test/dogs/dog.5804.jpg',
    'data/processed/test/cats/cat.11432.jpg',
    'data/processed/test/dogs/dog.9913.jpg',
)
with (OUTPUT_DIR / 'misclassified_samples.csv').open(
    encoding='utf-8-sig', newline=''
) as file:
    error_rows = {row['image_path']: row for row in csv.DictReader(file)}

figure, axes = plt.subplots(2, 3, figsize=(11, 7))
for number, (axis, relative_path) in enumerate(
    zip(axes.flat, selected_paths), start=1
):
    row = error_rows[relative_path]
    with Image.open(PROJECT_ROOT / relative_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(
        f'{number}. 真实{row["true_class"]} → 预测{row["predicted_class"]}'
        f'（{float(row["confidence"]) * 100:.1f}%）',
        fontsize=10,
    )
    axis.axis('off')
figure.suptitle('典型误判样本')
figure.tight_layout()
plt.show()

主要问题可以分成三类：

- Logo 和占位图没有真实动物，属于数据质量问题；
- 低光、局部特写和遮挡让模型拿不到完整信息；
- 哈士奇的尖耳和脸部轮廓与猫有部分相似，属于外观上的真实困难。

## 7. 第一层卷积核

第一层共有 8 个 `3×3×3` 卷积核。中灰色表示权重接近 0，颜色变化表示 RGB 通道上的正负权重。

In [ ]:
filters = model.features[0].weight.detach().cpu().numpy()
scale = np.abs(filters).max()
rgb_filters = np.clip(
    0.5 + filters.transpose(0, 2, 3, 1) / (2 * scale), 0, 1
)
norms = np.linalg.norm(filters.reshape(8, -1), axis=1)

figure, axes = plt.subplots(2, 4, figsize=(10, 5.5))
for index, axis in enumerate(axes.flat):
    axis.imshow(rgb_filters[index], interpolation='nearest')
    axis.set_title(f'Filter {index + 1}  ‖W‖={norms[index]:.3f}', fontsize=10)
    axis.set_xticks([])
    axis.set_yticks([])
figure.suptitle('训练后的第一层卷积核')
figure.tight_layout()
plt.show()

卷积核的正负权重交错，说明第一层主要在提取颜色差异、明暗变化和局部边缘。它们还不能直接表示“猫耳”或“狗鼻子”；第二层会继续组合这些底层特征，最后由全连接层完成分类。

## 当前结果

- 最佳验证准确率：**80.58%**；
- 测试准确率：**80.48%**；
- 猫判成狗：505 张；
- 狗判成猫：471 张。

下一步可以清理异常图片，并加入 Grad-CAM，观察模型做出判断时主要看了图片的哪些区域。